<a href="https://colab.research.google.com/github/Bindhya-K/ML-Practice/blob/main/Grad-Boost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [101]:
from sklearn.datasets import load_diabetes
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [102]:
diabetes = load_diabetes(as_frame=True)

In [104]:
diabetes.data.shape

(442, 10)

In [105]:
df = diabetes.frame
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [106]:
df.isnull().sum()

,0
age,0
sex,0
bmi,0
bp,0
s1,0
s2,0
s3,0
s4,0
s5,0
s6,0


In [107]:
X=diabetes.data
y=diabetes.target

In [120]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape

(353, 10)

In [109]:
X_test.shape

(89, 10)

In [110]:
from sklearn.ensemble import GradientBoostingRegressor
gradboost = GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,max_depth=8
                                      ,random_state=42)
gradboost.fit(X_train,y_train)
y_pred = gradboost.predict(X_test)
mean_squared_error(y_test,y_pred)

3553.0851141093303

In [113]:
# from scratch (without class)
X_train['pred0']=y_train.mean()
X_train['res0']=y_train-X_train['pred0']
tree1=DecisionTreeRegressor(max_depth=8,random_state=42)
tree1.fit(X_train.iloc[:,0:10],X_train['res0'])
X_train['pred1'] = X_train['pred0']+(0.1*tree1.predict(X_train.iloc[:,0:10]))
X_train['res1']=y_train-X_train['pred1']
tree2 = DecisionTreeRegressor(max_depth=8,random_state=42)
tree2.fit(X_train.iloc[:,0:10],X_train['res1'])
y_pred = y_train.mean()+(0.1*tree1.predict(X_test))+(0.1*tree2.predict(X_test))
mean_squared_error(y_test,y_pred)

4347.830021485996

In [121]:
# using own class
class grad_boost:
  def __init__(self,X,y,max_iteration=3):
    self.X=X
    self.y=y
    self.max_iteration=max_iteration

  def fit(self,X_train,y_train):
    self.X_train=X_train
    self.y_train=y_train
    self.regs=[]
    self.x_pred =[]
    for i in range(1,self.max_iteration):
      tree1 = DecisionTreeRegressor(max_depth=8)
      self.x_pred = self.y_train.mean() + sum(x.predict(X_train.values) for x in self.regs)
      tree1.fit(self.X_train.values,self.y_train-self.x_pred)

      self.regs.append(tree1)
      #X_train[f"pred{i}"]=y_train.mean()+tree1.predict(X_train.values)
      #X_train[f"res{i}"]=y_train-X_train[f"pred{i}"]

  def predict(self,X_test):
    self.X_test=X_test
    y_pred = self.y_train.mean()+sum(0.1 * regs.predict(X_test.values) for regs in self.regs)
    return y_pred



In [122]:
gb = grad_boost(X,y)

In [123]:
gb.fit(X_train,y_train)

In [124]:
gb.predict(X_test)

array([155.70455619, 142.76826586, 155.84372285, 158.85663952,
       152.61288952, 147.83788952, 162.03270519, 160.42538952,
       159.50058531, 156.94917984, 147.6528817 , 159.67742381,
       144.68172785, 164.01395519, 149.12915674, 144.90058531,
       168.56530215, 168.39863549, 162.5515992 , 155.09863549,
       151.45939745, 144.1278817 , 143.03899281, 156.75992581,
       150.03893053, 157.33446556, 166.42317523, 150.93761479,
       144.1278817 , 145.8765992 , 159.67742381, 143.03899281,
       154.452656  , 167.11288952, 143.72382142, 155.80288952,
       145.71993711, 144.25455619, 155.19030215, 147.8278817 ,
       144.1278817 , 146.86288952, 159.12904336, 147.60574666,
       155.61954836, 145.5778817 , 143.03899281, 152.75645312,
       144.1278817 , 158.8278817 , 149.8167083 , 145.5778817 ,
       158.2015992 , 152.75645312, 157.07122285, 149.8167083 ,
       143.86121503, 164.7536086 , 143.86121503, 143.86121503,
       154.60288952, 151.05872762, 152.10058531, 144.99

In [125]:
from sklearn.metrics import mean_squared_error
mean_squared_error(y_test,gb.predict(X_test))

4749.852303577947